# Day 2 Training: Healthcare Exploratory Data Analysis

This notebook covers practical Pandas and NumPy skills using a real-world healthcare dataset.

## Learning Goals
- Load and inspect healthcare data
- Use Pandas to clean and explore data
- GroupBy operations for patient insights
- Pivot tables for cross-sectional analysis
- Generate summary reports and exports

## 1. Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import os

# Load healthcare data
df = pd.read_csv('../Lab2_Healthcare_EDA/data/healthcare.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names and types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df.describe())

## 2. Data Cleaning

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Remove any duplicates
df_clean = df.drop_duplicates()
print(f"Shape after removing duplicates: {df_clean.shape}")

# Handle missing values (fill with median for numerical columns)
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print("\nAfter cleaning:")
print(df_clean.isnull().sum())

## 3. GroupBy Analysis

In [ ]:
# Group by condition
print("Patients by condition:")
by_condition = df_clean.groupby('condition').agg({
    'patient_id': 'count',
    'age': ['mean', 'min', 'max'],
    'total_cost': ['sum', 'mean'],
    'visits': 'mean'
})
print(by_condition)

# Group by department
print("\n\nAnalysis by department:")
by_dept = df_clean.groupby('department').agg({
    'patient_id': 'count',
    'total_cost': 'sum',
    'visits': 'mean',
    'age': 'mean'
}).round(2)
print(by_dept)

## 4. Age Group Analysis

In [ ]:
# Create age groups
age_bins = [0, 30, 45, 60, 100]
age_labels = ['Young (0-30)', 'Middle (31-45)', 'Senior (46-60)', 'Elderly (60+)']

df_clean['age_group'] = pd.cut(df_clean['age'], bins=age_bins, labels=age_labels, right=False)

# Analyze by age group
print("Analysis by age group:")
age_analysis = df_clean.groupby('age_group').agg({
    'patient_id': 'count',
    'total_cost': 'mean',
    'visits': 'mean',
    'condition': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
}).round(2)

age_analysis.columns = ['patient_count', 'avg_cost', 'avg_visits', 'common_condition']
print(age_analysis)

## 5. Pivot Table Analysis

In [ ]:
# Pivot: condition vs department
print("Patient count: Condition vs Department")
pivot_patients = df_clean.pivot_table(
    index='condition',
    columns='department',
    values='patient_id',
    aggfunc='count',
    fill_value=0
)
print(pivot_patients)

# Pivot: cost by condition and department
print("\n\nAverage cost: Condition vs Department")
pivot_cost = df_clean.pivot_table(
    index='condition',
    columns='department',
    values='total_cost',
    aggfunc='mean'
).round(2)
print(pivot_cost)

## 6. Summary Report

In [ ]:
# Overall summary
print("="*50)
print("HEALTHCARE EDA SUMMARY REPORT")
print("="*50)

print(f"\nTotal Patients: {df_clean['patient_id'].nunique()}")
print(f"Total Records: {len(df_clean)}")
print(f"Average Age: {df_clean['age'].mean():.1f} years")
print(f"Total Cost: ${df_clean['total_cost'].sum():,.2f}")
print(f"Average Cost per Patient: ${df_clean['total_cost'].mean():,.2f}")
print(f"Average Visits per Patient: {df_clean['visits'].mean():.2f}")

print(f"\nConditions in dataset: {', '.join(df_clean['condition'].unique())}")
print(f"Departments in dataset: {', '.join(df_clean['department'].unique())}")

# Top conditions by patient count
print("\nTop 5 conditions by patient count:")
top_conditions = df_clean['condition'].value_counts().head()
print(top_conditions)

# Top departments by cost
print("\nTotal cost by department:")
dept_cost = df_clean.groupby('department')['total_cost'].sum().sort_values(ascending=False)
print(dept_cost)